# MR MISLAV'S OPTION SCANNER REPO

https://github.com/MislavSag/options-scanner

# AAPL IV Slope Straddle Backtest

Backtest the IV slope straddle strategy using historical data from Polygon.io (Massive MCP).

**Strategy**: Compute constant-maturity IV at 30D and 180D tenors. Long straddle when IV slope is steep upward (backwardation), short straddle when inverted.

the connection to ibkr works (can buy/sell)

In [8]:
from ib_insync import IB, Stock, util, MarketOrder
import nest_asyncio
nest_asyncio.apply()

ib = IB()

try:
    ib.connect('127.0.0.1', 7497, clientId=1)  # 7497 = paper TWS
    print("Connected successfully!")
    print(f"Server version: {ib.client.serverVersion()}")
    print(f"Accounts: {ib.managedAccounts()}")

    # Quick market data test
    from ib_insync import Stock
    contract = Stock('AAPL', 'SMART', 'USD')
    bars = ib.reqHistoricalData(
        contract,
        endDateTime="",
        durationStr='1 Y',
        barSizeSetting='1 hour',
        whatToShow='TRADES',
        useRTH=True)
    ib.sleep(2)
    df = util.df(bars)

except Exception as e:
    print(f"Connection failed: {e}")

finally:
    ib.disconnect()

Connected successfully!
Server version: 176
Accounts: ['DUK874821']


In [ ]:
# buy order

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=1)  # 7497 = paper, 7496 = live

# Define the contract
contract = Stock('AAPL', 'SMART', 'USD')

# Place a market order for 1 share
order = MarketOrder('BUY', 1)
trade = ib.placeOrder(contract, order)

ib.sleep(2)  # Wait for fill

print(f"Order status: {trade.orderStatus.status}")
print(f"Filled @ {trade.orderStatus.avgFillPrice}")

ib.disconnect()

Order status: PendingSubmit
Filled @ 0.0


In [ ]:
# sell order

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=1)  # 7497 = paper, 7496 = live

# Define the contract
contract = Stock('AAPL', 'SMART', 'USD')

# Place a market order for 1 share
order = MarketOrder('SELL', 1)
trade = ib.placeOrder(contract, order)

ib.sleep(2)  # Wait for fill

print(f"Order status: {trade.orderStatus.status}")
print(f"Filled @ {trade.orderStatus.avgFillPrice}")

ib.disconnect()

Error 10349, reqId 6: Order TIF was set to DAY based on order preset.
Canceled order: Trade(contract=Stock(symbol='AAPL', exchange='SMART', currency='USD'), order=MarketOrder(orderId=6, clientId=1, action='SELL', totalQuantity=1), orderStatus=OrderStatus(orderId=6, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 5, 17, 19, 25, 921707, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 5, 17, 19, 26, 144442, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 6: Order TIF was set to DAY based on order preset.', errorCode=10349)], advancedError='')


Order status: Filled
Filled @ 259.13


---

---

---

---

---

---

---

In [ ]:
# Cell 0: Configuration & Imports
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
from datetime import datetime, date, timedelta
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──
SYMBOL = "AAPL"
BACKTEST_START = "2025-03-05"
BACKTEST_END = "2026-03-05"

# Strategy parameters (from config.yaml)
TENORS = [30, 180]
TARGET_DTE = 30
MIN_DTE = 14
MAX_DTE = 60
HOLD_DAYS = None  # None = hold to expiry

# IV slope signal thresholds (for single-stock absolute threshold)
IV_SLOPE_LONG_THRESHOLD = 0.10   # slope > 0.10 => long straddle
IV_SLOPE_SHORT_THRESHOLD = -0.05 # slope < -0.05 => short straddle

# Filtering (from iv_calculator.py)
MIN_ABS_DELTA = 0.35
MAX_ABS_DELTA = 0.65
MIN_IV = 0.03
MAX_IV = 2.0

# Position sizing
CAPITAL = 100_000
CONTRACTS_PER_TRADE = 1

# Data paths
DATA_DIR = "data/backtest"
STOCK_FILE = f"{DATA_DIR}/aapl_stock_daily.csv"
OPTIONS_FILE = f"{DATA_DIR}/aapl_options_daily.csv"
YIELDS_FILE = f"{DATA_DIR}/treasury_yields.csv"

print(f"Backtest: {SYMBOL} from {BACKTEST_START} to {BACKTEST_END}")
print(f"Tenors: {TENORS}, Target DTE: {TARGET_DTE}")
print(f"IV slope thresholds: long>{IV_SLOPE_LONG_THRESHOLD}, short<{IV_SLOPE_SHORT_THRESHOLD}")

In [ ]:
# Cell 1: Load Pre-Fetched Data

# Stock daily prices
stock_df = pd.read_csv(STOCK_FILE, parse_dates=["date"])
stock_df = stock_df.sort_values("date").reset_index(drop=True)
print(f"Stock data: {len(stock_df)} days, {stock_df['date'].min().date()} to {stock_df['date'].max().date()}")

# Options daily OHLCV
options_df = pd.read_csv(OPTIONS_FILE)
options_df["date"] = pd.to_datetime(options_df["date"])
options_df["expiry_date"] = pd.to_datetime(options_df["expiry"], format="%Y%m%d")
print(f"Options data: {len(options_df)} rows, {options_df['option_ticker'].nunique()} unique contracts")

# Treasury yields (risk-free rate)
yields_df = pd.read_csv(YIELDS_FILE, parse_dates=["date"])
yields_df = yields_df.sort_values("date").reset_index(drop=True)
yields_df["rf_rate"] = yields_df["yield_3_month"] / 100.0
print(f"Yields data: {len(yields_df)} rows")

# Merge stock close + risk-free rate by date
daily_df = stock_df[["date", "close"]].rename(columns={"close": "stock_close"})
daily_df = daily_df.merge(yields_df[["date", "rf_rate"]], on="date", how="left")
daily_df["rf_rate"] = daily_df["rf_rate"].ffill().bfill()
print(f"Daily reference: {len(daily_df)} rows")
daily_df.head()

In [ ]:
# Cell 2: Black-Scholes Functions

def bs_price(S, K, T, r, sigma, option_type="C"):
    """Black-Scholes European option price."""
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0) if option_type == "C" else max(K - S, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "C":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def bs_delta(S, K, T, r, sigma, option_type="C"):
    """Black-Scholes delta."""
    if T <= 0 or sigma <= 0:
        if option_type == "C":
            return 1.0 if S > K else 0.0
        return -1.0 if S < K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1) if option_type == "C" else norm.cdf(d1) - 1.0


def implied_volatility(market_price, S, K, T, r, option_type="C",
                        lower=0.001, upper=5.0):
    """Solve for IV using Brent's method. Returns NaN on failure."""
    if T <= 1e-6 or market_price <= 0:
        return np.nan
    intrinsic = max(S - K, 0.0) if option_type == "C" else max(K - S, 0.0)
    if market_price <= intrinsic + 1e-8:
        return np.nan
    def objective(sigma):
        return bs_price(S, K, T, r, sigma, option_type) - market_price
    try:
        f_lo, f_hi = objective(lower), objective(upper)
        if f_lo * f_hi > 0:
            return np.nan
        return brentq(objective, lower, upper, xtol=1e-6)
    except (ValueError, RuntimeError):
        return np.nan


# Sanity check
test_iv = implied_volatility(10.0, 200.0, 200.0, 30/365, 0.04, "C")
print(f"Sanity: IV for $10 ATM call, 30 DTE, S=200: {test_iv:.4f} ({test_iv*100:.1f}%)")

In [ ]:
# Cell 3: Compute Historical IV & Greeks for All Options Data

def compute_historical_iv(options_df, daily_df):
    """For each option-day observation, compute IV and delta via Black-Scholes."""
    df = options_df.merge(daily_df, on="date", how="inner")
    df["dte_days"] = (df["expiry_date"] - df["date"]).dt.days
    df["T"] = df["dte_days"] / 365.0
    df["option_mid"] = df["close"]  # use option daily close

    # Vectorized would be ideal but IV solver needs iteration
    ivs, deltas = [], []
    for _, row in df.iterrows():
        iv = implied_volatility(
            row["option_mid"], row["stock_close"], row["strike"],
            row["T"], row["rf_rate"], row["contract_type"]
        )
        ivs.append(iv)
        if np.isfinite(iv):
            deltas.append(bs_delta(
                row["stock_close"], row["strike"], row["T"],
                row["rf_rate"], iv, row["contract_type"]
            ))
        else:
            deltas.append(np.nan)

    df["iv"] = ivs
    df["delta"] = deltas

    # Apply filters (mirrors iv_calculator.py filter_options)
    valid = df[
        df["iv"].between(MIN_IV, MAX_IV) &
        df["delta"].abs().between(MIN_ABS_DELTA, MAX_ABS_DELTA) &
        (df["option_mid"] > 0) &
        (df["dte_days"] > 0)
    ].copy()

    print(f"Total option-day observations: {len(df)}")
    print(f"After IV/delta filtering: {len(valid)} ({len(valid)/max(len(df),1)*100:.1f}%)")
    if len(valid) > 0:
        print(f"IV range: {valid['iv'].min():.3f} to {valid['iv'].max():.3f}")
        print(f"Delta range: {valid['delta'].min():.3f} to {valid['delta'].max():.3f}")
    return valid

options_with_iv = compute_historical_iv(options_df, daily_df)
options_with_iv.head(10)

In [ ]:
# Cell 4: Build ATM Straddle IV Term Structure

def build_daily_atm_iv(options_with_iv):
    """Per trading day x expiry: find ATM strike, compute straddle IV.
    Mirrors iv_calculator.py find_atm_options + compute_straddle_iv."""
    results = []
    for eval_date, day_group in options_with_iv.groupby("date"):
        stock_price = day_group["stock_close"].iloc[0]
        for expiry, exp_grp in day_group.groupby("expiry_date"):
            # Find ATM strike
            atm_diff = (exp_grp["strike"] - stock_price).abs()
            atm_strike = exp_grp.loc[atm_diff.idxmin(), "strike"]
            atm = exp_grp[exp_grp["strike"] == atm_strike]

            call_iv = atm.loc[atm["contract_type"] == "C", "iv"]
            put_iv = atm.loc[atm["contract_type"] == "P", "iv"]
            if call_iv.empty or put_iv.empty:
                continue

            atm_iv = (call_iv.values[0] + put_iv.values[0]) / 2
            dte = (expiry - eval_date).days
            if dte <= 0 or not np.isfinite(atm_iv):
                continue

            results.append({
                "eval_date": eval_date, "expiry_date": expiry,
                "dte_days": dte, "atm_strike": atm_strike,
                "stock_price": stock_price,
                "call_iv": call_iv.values[0], "put_iv": put_iv.values[0],
                "atm_iv": atm_iv,
            })

    atm_df = pd.DataFrame(results)
    print(f"ATM IV observations: {len(atm_df)} across {atm_df['eval_date'].nunique()} dates")
    return atm_df

atm_iv_df = build_daily_atm_iv(options_with_iv)
atm_iv_df.head(10)

In [ ]:
# Cell 5: Interpolate Constant Maturity IV (30D & 180D)

def interpolate_cm_iv(atm_iv_df, tenors=TENORS):
    """Interpolate ATM IV to constant maturity tenors.
    Mirrors iv_calculator.py interpolate_constant_maturity."""
    results = []
    for eval_date, grp in atm_iv_df.groupby("eval_date"):
        grp = grp.sort_values("dte_days")
        x = grp["dte_days"].values.astype(float)
        y = grp["atm_iv"].values.astype(float)

        mask = np.isfinite(x) & np.isfinite(y)
        x, y = x[mask], y[mask]

        row = {"eval_date": eval_date, "stock_price": grp["stock_price"].iloc[0]}

        if len(x) == 0:
            for t in tenors:
                row[f"IV{t}D"] = np.nan
        elif len(np.unique(x)) < 2:
            for t in tenors:
                row[f"IV{t}D"] = y[0]
        else:
            ux, idx = np.unique(x, return_inverse=True)
            uy = np.array([y[idx == i].mean() for i in range(len(ux))])
            for t in tenors:
                row[f"IV{t}D"] = float(np.interp(t, ux, uy))

        results.append(row)

    cm_df = pd.DataFrame(results)
    cm_df = cm_df.dropna(subset=[f"IV{t}D" for t in tenors])

    # IV slope = (IV180D - IV30D) / IV180D
    cm_df["iv_slope"] = np.where(
        cm_df["IV180D"] > 0,
        (cm_df["IV180D"] - cm_df["IV30D"]) / cm_df["IV180D"],
        np.nan,
    )
    cm_df = cm_df[np.isfinite(cm_df["iv_slope"])].copy()

    print(f"Constant maturity IV: {len(cm_df)} trading days")
    print(f"IV30D: {cm_df['IV30D'].min()*100:.1f}% to {cm_df['IV30D'].max()*100:.1f}%")
    print(f"IV180D: {cm_df['IV180D'].min()*100:.1f}% to {cm_df['IV180D'].max()*100:.1f}%")
    print(f"IV slope: {cm_df['iv_slope'].min():.4f} to {cm_df['iv_slope'].max():.4f}")
    return cm_df

cm_iv_df = interpolate_cm_iv(atm_iv_df)
cm_iv_df.head(10)

In [ ]:
# Cell 6: Visualize IV Term Structure & Slope

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# IV30D vs IV180D
axes[0].plot(cm_iv_df["eval_date"], cm_iv_df["IV30D"] * 100, label="IV30D", linewidth=1.2)
axes[0].plot(cm_iv_df["eval_date"], cm_iv_df["IV180D"] * 100, label="IV180D", linewidth=1.2)
axes[0].set_ylabel("Implied Volatility (%)")
axes[0].set_title(f"{SYMBOL} Constant Maturity IV (30D vs 180D)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IV Slope with thresholds
axes[1].plot(cm_iv_df["eval_date"], cm_iv_df["iv_slope"], color="purple", linewidth=1.2)
axes[1].axhline(y=IV_SLOPE_LONG_THRESHOLD, color="green", linestyle="--", alpha=0.7,
                label=f"Long threshold ({IV_SLOPE_LONG_THRESHOLD})")
axes[1].axhline(y=IV_SLOPE_SHORT_THRESHOLD, color="red", linestyle="--", alpha=0.7,
                label=f"Short threshold ({IV_SLOPE_SHORT_THRESHOLD})")
axes[1].set_ylabel("IV Slope")
axes[1].set_title("IV Slope = (IV180D - IV30D) / IV180D")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Stock price
axes[2].plot(stock_df["date"], stock_df["close"], color="black", linewidth=1.2)
axes[2].set_ylabel("Stock Price ($)")
axes[2].set_title(f"{SYMBOL} Stock Price")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Signal Generation

def generate_signals(cm_iv_df):
    """Generate entry signals based on IV slope thresholds.
    Single-stock version of analyze_scanned_options.py _assign_signals."""
    signals = cm_iv_df.copy()
    signals["signal"] = "none"
    signals.loc[signals["iv_slope"] >= IV_SLOPE_LONG_THRESHOLD, "signal"] = "long_straddle"
    signals.loc[signals["iv_slope"] <= IV_SLOPE_SHORT_THRESHOLD, "signal"] = "short_straddle"

    # Only signal on regime transitions (first day of new signal)
    signals["prev_signal"] = signals["signal"].shift(1)
    signals["entry_signal"] = np.where(
        (signals["signal"] != signals["prev_signal"]) & (signals["signal"] != "none"),
        signals["signal"], "none"
    )

    n_long = (signals["entry_signal"] == "long_straddle").sum()
    n_short = (signals["entry_signal"] == "short_straddle").sum()
    print(f"Entry signals: {n_long} long straddles, {n_short} short straddles")
    return signals

signals_df = generate_signals(cm_iv_df)
signals_df[signals_df["entry_signal"] != "none"][
    ["eval_date", "stock_price", "IV30D", "IV180D", "iv_slope", "entry_signal"]
]

In [ ]:
# Cell 8: Backtest Engine

class StraddleBacktest:
    """Event-driven backtester for IV slope straddle strategy.
    Entry: buy/sell ATM straddle at ~30 DTE when signal fires.
    Exit: hold to expiry (intrinsic) or configurable hold period."""

    def __init__(self, options_with_iv, daily_df, signals_df):
        self.options = options_with_iv
        self.daily = daily_df
        self.signals = signals_df
        self.trades = []

    def find_straddle_entry(self, eval_date, stock_price):
        """Find best ATM straddle near target DTE.
        Mirrors analyze_scanned_options.py _pick_contract_per_symbol."""
        day_opts = self.options[self.options["date"] == eval_date]
        if day_opts.empty:
            return None
        candidates = day_opts[day_opts["dte_days"].between(MIN_DTE, MAX_DTE)]
        if candidates.empty:
            return None

        best, best_score = None, float("inf")
        for expiry, exp_grp in candidates.groupby("expiry_date"):
            atm_diff = (exp_grp["strike"] - stock_price).abs()
            atm_strike = exp_grp.loc[atm_diff.idxmin(), "strike"]
            atm_calls = exp_grp[(exp_grp["strike"] == atm_strike) & (exp_grp["contract_type"] == "C")]
            atm_puts = exp_grp[(exp_grp["strike"] == atm_strike) & (exp_grp["contract_type"] == "P")]
            if atm_calls.empty or atm_puts.empty:
                continue
            call_row, put_row = atm_calls.iloc[0], atm_puts.iloc[0]
            dte = call_row["dte_days"]
            score = abs(dte - TARGET_DTE) + 0.01 * abs(atm_strike - stock_price)
            if score < best_score:
                best_score = score
                best = {
                    "expiry_date": expiry, "strike": atm_strike, "dte_at_entry": dte,
                    "call_price": call_row["option_mid"], "put_price": put_row["option_mid"],
                    "call_iv": call_row["iv"], "put_iv": put_row["iv"],
                    "call_ticker": call_row["option_ticker"], "put_ticker": put_row["option_ticker"],
                }
        return best

    def get_option_price_on_date(self, ticker, target_date, strike, ctype, stock_price):
        """Get option price on date, or intrinsic at expiry."""
        row = self.options[
            (self.options["option_ticker"] == ticker) & (self.options["date"] == target_date)
        ]
        if not row.empty:
            return row.iloc[0]["option_mid"]
        return max(stock_price - strike, 0.0) if ctype == "C" else max(strike - stock_price, 0.0)

    def run(self):
        open_positions = []
        trading_dates = sorted(self.signals["eval_date"].unique())

        for eval_date in trading_dates:
            eval_ts = pd.Timestamp(eval_date)
            stock_row = self.daily[self.daily["date"] == eval_ts]
            if stock_row.empty:
                continue
            stock_price = stock_row.iloc[0]["stock_close"]

            # Check exits
            still_open = []
            for pos in open_positions:
                should_exit = False
                reason = None
                if eval_ts >= pos["expiry_date"]:
                    should_exit, reason = True, "expiry"
                elif HOLD_DAYS and (eval_ts - pos["entry_date"]).days >= HOLD_DAYS:
                    should_exit, reason = True, "hold_limit"

                if should_exit:
                    call_exit = self.get_option_price_on_date(
                        pos["call_ticker"], eval_ts, pos["strike"], "C", stock_price)
                    put_exit = self.get_option_price_on_date(
                        pos["put_ticker"], eval_ts, pos["strike"], "P", stock_price)
                    exit_straddle = call_exit + put_exit
                    entry_straddle = pos["entry_straddle_price"]
                    multiplier = 1 if pos["side"] == "long_straddle" else -1
                    pnl = multiplier * (exit_straddle - entry_straddle) * 100 * CONTRACTS_PER_TRADE
                    pos.update({
                        "exit_date": eval_ts, "exit_straddle_price": exit_straddle,
                        "exit_reason": reason, "pnl": pnl,
                        "return_pct": pnl / (entry_straddle * 100 * CONTRACTS_PER_TRADE) * 100,
                    })
                    self.trades.append(pos)
                else:
                    still_open.append(pos)
            open_positions = still_open

            # Check entries
            sig_row = self.signals[self.signals["eval_date"] == eval_ts]
            if sig_row.empty:
                continue
            entry_signal = sig_row.iloc[0]["entry_signal"]
            if entry_signal == "none" or open_positions:
                continue

            straddle = self.find_straddle_entry(eval_ts, stock_price)
            if straddle is None:
                continue
            entry_price = straddle["call_price"] + straddle["put_price"]
            if entry_price <= 0:
                continue

            open_positions.append({
                "entry_date": eval_ts, "side": entry_signal,
                "strike": straddle["strike"], "expiry_date": straddle["expiry_date"],
                "dte_at_entry": straddle["dte_at_entry"],
                "stock_price_at_entry": stock_price,
                "entry_straddle_price": entry_price,
                "call_price_entry": straddle["call_price"],
                "put_price_entry": straddle["put_price"],
                "call_ticker": straddle["call_ticker"],
                "put_ticker": straddle["put_ticker"],
                "iv_slope": sig_row.iloc[0]["iv_slope"],
            })

        # Close remaining positions at last date
        if open_positions:
            last_date = pd.Timestamp(trading_dates[-1])
            last_stock = self.daily[self.daily["date"] == last_date]
            if not last_stock.empty:
                lp = last_stock.iloc[0]["stock_close"]
                for pos in open_positions:
                    ce = self.get_option_price_on_date(pos["call_ticker"], last_date, pos["strike"], "C", lp)
                    pe = self.get_option_price_on_date(pos["put_ticker"], last_date, pos["strike"], "P", lp)
                    es = ce + pe
                    ep = pos["entry_straddle_price"]
                    mult = 1 if pos["side"] == "long_straddle" else -1
                    pnl = mult * (es - ep) * 100 * CONTRACTS_PER_TRADE
                    pos.update({"exit_date": last_date, "exit_straddle_price": es,
                                "exit_reason": "backtest_end", "pnl": pnl,
                                "return_pct": pnl / (ep * 100 * CONTRACTS_PER_TRADE) * 100})
                    self.trades.append(pos)

        return pd.DataFrame(self.trades)


bt = StraddleBacktest(options_with_iv, daily_df, signals_df)
trades_df = bt.run()
print(f"Total trades: {len(trades_df)}")
trades_df

In [ ]:
# Cell 9: Performance Analytics

def compute_performance(trades_df, capital=CAPITAL):
    if trades_df.empty:
        print("No trades to analyze.")
        return {}

    total_pnl = trades_df["pnl"].sum()
    n_trades = len(trades_df)
    n_win = (trades_df["pnl"] > 0).sum()
    n_lose = (trades_df["pnl"] < 0).sum()
    win_rate = n_win / n_trades * 100
    avg_win = trades_df.loc[trades_df["pnl"] > 0, "pnl"].mean() if n_win else 0
    avg_loss = trades_df.loc[trades_df["pnl"] < 0, "pnl"].mean() if n_lose else 0
    pf = abs(avg_win * n_win / (avg_loss * n_lose)) if n_lose and avg_loss != 0 else float("inf")

    long_t = trades_df[trades_df["side"] == "long_straddle"]
    short_t = trades_df[trades_df["side"] == "short_straddle"]

    metrics = {
        "Total P&L": f"${total_pnl:,.2f}",
        "Total Return": f"{total_pnl / capital * 100:.2f}%",
        "Trades": n_trades,
        "Win Rate": f"{win_rate:.1f}%",
        "Avg Win": f"${avg_win:,.2f}",
        "Avg Loss": f"${avg_loss:,.2f}",
        "Profit Factor": f"{pf:.2f}",
        "Max Win": f"${trades_df['pnl'].max():,.2f}",
        "Max Loss": f"${trades_df['pnl'].min():,.2f}",
        "Long Trades": f"{len(long_t)} (P&L: ${long_t['pnl'].sum():,.2f})" if len(long_t) else "0",
        "Short Trades": f"{len(short_t)} (P&L: ${short_t['pnl'].sum():,.2f})" if len(short_t) else "0",
    }
    print("=" * 50)
    print("BACKTEST PERFORMANCE SUMMARY")
    print("=" * 50)
    for k, v in metrics.items():
        print(f"  {k:.<30} {v}")
    return metrics

metrics = compute_performance(trades_df)

In [ ]:
# Cell 10: Equity Curve & Trade Visualization

if not trades_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Cumulative P&L
    ts = trades_df.sort_values("exit_date")
    ts["cum_pnl"] = ts["pnl"].cumsum()
    axes[0, 0].plot(ts["exit_date"], ts["cum_pnl"], marker="o", linewidth=1.5)
    axes[0, 0].axhline(y=0, color="black", linestyle="-", alpha=0.3)
    axes[0, 0].set_title("Cumulative P&L")
    axes[0, 0].set_ylabel("P&L ($)")
    axes[0, 0].grid(True, alpha=0.3)

    # Per-trade P&L
    colors = ["green" if p > 0 else "red" for p in ts["pnl"]]
    axes[0, 1].bar(range(len(ts)), ts["pnl"], color=colors, alpha=0.7)
    axes[0, 1].set_title("Per-Trade P&L")
    axes[0, 1].set_xlabel("Trade #")
    axes[0, 1].set_ylabel("P&L ($)")
    axes[0, 1].grid(True, alpha=0.3)

    # Stock price with trade markers
    axes[1, 0].plot(stock_df["date"], stock_df["close"], color="gray", alpha=0.5, linewidth=1)
    for _, trade in trades_df.iterrows():
        color = "green" if trade["side"] == "long_straddle" else "red"
        marker = "^" if trade["side"] == "long_straddle" else "v"
        axes[1, 0].scatter(trade["entry_date"], trade["stock_price_at_entry"],
                           color=color, marker=marker, s=100, zorder=5)
    axes[1, 0].set_title(f"{SYMBOL} with Trade Entries (green=long, red=short)")
    axes[1, 0].set_ylabel("Stock Price ($)")
    axes[1, 0].grid(True, alpha=0.3)

    # IV slope with signal zones
    axes[1, 1].plot(cm_iv_df["eval_date"], cm_iv_df["iv_slope"], color="purple", linewidth=1)
    axes[1, 1].fill_between(cm_iv_df["eval_date"], IV_SLOPE_LONG_THRESHOLD,
                             cm_iv_df["iv_slope"].max() * 1.1, alpha=0.1, color="green", label="Long zone")
    axes[1, 1].fill_between(cm_iv_df["eval_date"], cm_iv_df["iv_slope"].min() * 1.1,
                             IV_SLOPE_SHORT_THRESHOLD, alpha=0.1, color="red", label="Short zone")
    axes[1, 1].set_title("IV Slope with Signal Zones")
    axes[1, 1].set_ylabel("IV Slope")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No trades to visualize.")

In [ ]:
# Cell 11: Trade Log

if not trades_df.empty:
    display_cols = [
        "entry_date", "exit_date", "side", "strike", "dte_at_entry",
        "stock_price_at_entry", "iv_slope",
        "entry_straddle_price", "exit_straddle_price",
        "exit_reason", "pnl", "return_pct",
    ]
    disp = trades_df[display_cols].copy()
    disp["entry_date"] = disp["entry_date"].dt.strftime("%Y-%m-%d")
    disp["exit_date"] = disp["exit_date"].dt.strftime("%Y-%m-%d")
    disp["pnl"] = disp["pnl"].map("${:,.2f}".format)
    disp["return_pct"] = disp["return_pct"].map("{:.1f}%".format)
    disp["entry_straddle_price"] = disp["entry_straddle_price"].map("${:.2f}".format)
    disp["exit_straddle_price"] = disp["exit_straddle_price"].map("${:.2f}".format)
    disp["iv_slope"] = disp["iv_slope"].map("{:.4f}".format)
    print("TRADE LOG")
    print("=" * 120)
    display(disp)
else:
    print("No trades.")

In [ ]:
# Cell 12: Sensitivity Analysis - Sweep IV slope thresholds

results = []
for lt in np.arange(0.05, 0.30, 0.05):
    for st in np.arange(-0.15, 0.05, 0.05):
        temp = cm_iv_df.copy()
        temp["signal"] = "none"
        temp.loc[temp["iv_slope"] >= lt, "signal"] = "long_straddle"
        temp.loc[temp["iv_slope"] <= st, "signal"] = "short_straddle"
        temp["prev_signal"] = temp["signal"].shift(1)
        temp["entry_signal"] = np.where(
            (temp["signal"] != temp["prev_signal"]) & (temp["signal"] != "none"),
            temp["signal"], "none"
        )
        b = StraddleBacktest(options_with_iv, daily_df, temp)
        t = b.run()
        if not t.empty:
            results.append({
                "long_thresh": round(lt, 2), "short_thresh": round(st, 2),
                "n_trades": len(t), "total_pnl": t["pnl"].sum(),
                "win_rate": (t["pnl"] > 0).mean() * 100,
            })

if results:
    sens_df = pd.DataFrame(results)
    print("Threshold Sensitivity Analysis (Top 10 by P&L):")
    display(sens_df.sort_values("total_pnl", ascending=False).head(10))
else:
    print("No results from sensitivity sweep.")